In [1]:
import pandas as pd

In [2]:
import numpy as np 

In [3]:
df = pd.read_csv('../data/raw_data/stock_market.csv')

In [4]:
cleaned_df = df.copy()

In [5]:
pd.set_option("display.max_columns",None)

In [6]:
cleaned_df.shape

(20914, 28)

In [7]:
cleaned_df.duplicated().sum()

np.int64(124)

In [8]:
cleaned_df=cleaned_df.drop_duplicates()

In [9]:
cleaned_df.shape

(20790, 28)

In [10]:
cleaned_df.groupby('company_id')['company_name'].nunique().max()

np.int64(1)

### Duplicate Removal
* **Action:** Dropped 124 exact duplicate row using `drop_duplicates()`.
* **Justification:** Retaining duplicate trading records inflates transaction volumes and skew price aggregations in SQL and Power BI.

In [11]:
cleaned_df['close_price'].unique()[:25]

<StringArray>
[    '913.34',     '638.64',      '109.0', 'Rs. 1470.6',    '2057.69',
     '106.15',    'unknown',    '5041.81',    '2925.02',      '462.0',
     '699.44', 'Rs. 100.24',     '230.18',     '608.31',     '117.01',
   '₹2203.53',    '3716.58',    '5805.85',    '11219.6',     '112.62',
     '1689.3',    '1217.95',   '₹1931.74',    '4231.66',    '1338.26']
Length: 25, dtype: str

In [12]:
dirty_rows = cleaned_df[cleaned_df['close_price'].astype(str).str.contains(r'[a-zA-Z\$]',regex =True)]
dirty_rows['close_price'].head(20)

3       Rs. 1470.6
6          unknown
7          unknown
12      Rs. 100.24
17         unknown
31       Rs. 99.97
32     Rs. 1397.67
43     Rs. 3244.27
48     Rs. 1086.35
55     Rs. 1253.78
76     Rs. 3477.91
80      Rs. 425.89
83     Rs. 1137.28
85      Rs. 100.62
95      Rs. 650.77
109        unknown
114    Rs. 2379.02
122        unknown
143        unknown
146     Rs. 458.78
Name: close_price, dtype: str

In [13]:
cleaned_df['close_price']  = (
    cleaned_df['close_price']
    .astype(str)
    .str.replace('rs\.?','',case=False,regex=True)
    .str.replace('₹','',regex=False)
    .str.replace('$','',regex=False)
    .str.strip()
)

<>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\DELL\AppData\Local\Temp\ipykernel_18608\461253527.py:4: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  .str.replace('rs\.?','',case=False,regex=True)


In [14]:
cleaned_df['close_price'] = pd.to_numeric(cleaned_df['close_price'],errors='coerce')

In [15]:
cleaned_df['close_price'].isnull().sum()

np.int64(1016)

In [16]:
cleaned_df = cleaned_df.dropna(subset=['close_price'])

In [17]:
cleaned_df.shape

(19774, 28)

In [18]:
invalid_price = (cleaned_df['close_price'] <= 0).sum()
print('invalid price count:',invalid_price)

invalid price count: 0


### Range validation: `close_price`
* **Action:** verified that all `close_price` records are strictly positive(>0).
* Close price columns have been cleaned removing `Rs.`,`₹`,and converted to `float64

## Missing values treatments
* **close_price**: Removed 1016 rows with missing prices to ensure accurate technical indicator calculations and reliable filtering.

In [19]:
cleaned_df[['delivery_pct','atr_14']].isnull().sum()

delivery_pct    2976
atr_14           955
dtype: int64

In [20]:
cols_to_clean = ['delivery_pct','atr_14']
for col in cols_to_clean:
    cleaned_df[col]=(cleaned_df[col]
                     .astype(str)
                     .str.replace(r'(?i)%|pts?\.?','',regex=True)
                     .str.strip()
                     )
    cleaned_df[col] = pd.to_numeric(cleaned_df[col],errors='coerce')

In [21]:
cleaned_df[['delivery_pct','atr_14']].isnull().sum()

delivery_pct    2976
atr_14           955
dtype: int64

In [22]:
invalid_delivery = (cleaned_df['delivery_pct']<0 | (cleaned_df['delivery_pct']>100))
print(invalid_delivery.sum())

0


In [23]:
invalid_atr = cleaned_df['atr_14']<0
print(invalid_atr.sum())

0


In [24]:
print((cleaned_df['atr_14'] == 0).sum())

10


In [25]:
cleaned_df=cleaned_df[cleaned_df['atr_14'].isna() | (cleaned_df['atr_14']>0)]

* **Atr cleaning:** Removed 10 invalid zero-value rows.
* **Objective** prevent potential `zeroDivisionError` during technical analysis.

In [26]:
cleaned_df.shape

(19764, 28)

In [27]:
cleaned_df[['delivery_pct','atr_14','close_price']].describe()

,delivery_pct,atr_14,close_price
count,16789.000000,18809.000000,19764.000000
mean,44.855268,35.467772,2117.532180
std,11.992640,58.970637,2301.196554
min,10.000000,0.010000,56.980000
25%,36.700000,3.750000,537.757500
50%,44.900000,14.370000,1396.885000
75%,53.000000,41.540000,2968.757500
max,89.100000,1305.100000,13674.830000


  ## missing value treatment
 * **action :** invalid entries converted to 'NaN' using 'pd.to_numeric'(errors='coerce')' and retained as missing.
 * **justification :** Imputing(0/mean) distort delivery and volatility metrics, while dropping rows loses useful data.
 * Removed strip `%`/`pts` suffixes from `delivery_pct` and `atr_14`

In [28]:
cleaned_df[['sector','exchange','market_cap_segment','settlement_mode','settlement_type','data_source','trading_status']].isna().sum()

sector                    0
exchange                  0
market_cap_segment      973
settlement_mode        1218
settlement_type           0
data_source            2308
trading_status        10793
dtype: int64

In [29]:
cleaned_df[['sector','exchange','market_cap_segment','settlement_mode','settlement_type','data_source','trading_status','next_day_direction']].dtypes

sector                str
exchange              str
market_cap_segment    str
settlement_mode       str
settlement_type       str
data_source           str
trading_status        str
next_day_direction    str
dtype: object

In [30]:
cat_cols = ['sector','exchange','market_cap_segment','settlement_mode','settlement_type','data_source','trading_status','next_day_direction']
for col in cat_cols:
    if col in cleaned_df.columns:
        cleaned_df[col]=cleaned_df[col].str.strip().str.title()
        if 'exchange' in cleaned_df.columns:
            cleaned_df['exchange']=cleaned_df['exchange'].str.upper()
            cleaned_df[cat_cols] = cleaned_df[cat_cols].fillna('Unknown')


## Missing values in categorical columns

* Action : Replaced missing values(`NaN`) with `Unknown` across text columns(`sector`,`exchange`,`data_source`,etc.).
* **Case normalization done on categorical columns `sector`, `exchange`,`market_cap_segment`,`settlement_mode`,`settlement_type`,`data_source`,`trading_status`.
* ** Why this approach ?**
keeps useful data intact: Deleting these rows just because a text label  is missing would throw away perfectly valid price and volume numbers.
* Cleaner SQL and Power BI reports: blank values look awakward in charts and slicers labeling them as `Unknown`  keeps filters neat and makes it easy to track unclassified records.

In [31]:
cleaned_df[['sector','exchange','settlement_mode','market_cap_segment','settlement_type','data_source','trading_status']].head(10)

,sector,exchange,settlement_mode,market_cap_segment,settlement_type,data_source,trading_status
0,Private Sector Bank,NSE,F&O,Mid Cap,T+1,Moneycontrol,Suspended
1,Public Sector Bank,NSE,Cash,Large Cap,Delivery,Nse Bhavcopy,Unknown
2,Private Bank,NSE,Cash,Mid Cap,Delivery,Bloomberg,Unknown
3,It Services,NSE,F&O,Large Cap,T+1,Unknown,Unknown
4,It Services,NSE,Cash,Mid Cap,Btst,Bloomberg,Active
5,Private Bank,NSE,Cash,Mid Cap,Intraday,Bloomberg,Unknown
8,It Services,NSE,F&O,Mid Cap,T+1,Reuters,Suspended
9,It Services,NATIONAL STOCK EXCHANGE,Cash,Mid Cap,T+1,Reuters,Available
10,It Services,NSE,Margin,Large Cap,Intraday,Reuters,Unknown
11,Sfb,NSE,Cash,Mid Cap,Delivery,Reuters,Unknown


In [32]:
cleaned_df['sector'].unique()

<StringArray>
[   'Private Sector Bank',     'Public Sector Bank',           'Private Bank',
            'It Services',                    'Sfb', 'Information Technology',
               'Psu Bank',     'Small Finance Bank']
Length: 8, dtype: str

In [33]:
cleaned_df['exchange'].unique()

<StringArray>
['NSE', 'NATIONAL STOCK EXCHANGE']
Length: 2, dtype: str

In [34]:
cleaned_df['settlement_mode'].unique()

<StringArray>
['F&O', 'Cash', 'Margin', 'Unknown']
Length: 4, dtype: str

In [35]:
cleaned_df['market_cap_segment'].unique()

<StringArray>
['Mid Cap', 'Large Cap', 'Unknown']
Length: 3, dtype: str

In [36]:
cleaned_df['settlement_type'].unique()

<StringArray>
['T+1', 'Delivery', 'Btst', 'Intraday']
Length: 4, dtype: str

In [37]:
cleaned_df['data_source'].unique()

<StringArray>
['Moneycontrol', 'Nse Bhavcopy', 'Bloomberg', 'Unknown', 'Reuters']
Length: 5, dtype: str

In [38]:
cleaned_df['trading_status'].unique()

<StringArray>
['Suspended', 'Unknown', 'Active', 'Available', 'Halted']
Length: 5, dtype: str

In [39]:
Sector_map = {
    'Private Sector Bank':'Private Bank',
    'Psu Bank':'Public Sector Bank',
    'Sfb':'Small Finance Bank',
    'Information Technology':'IT Services',
    'It Services':'IT Services',
    'IT Service':'IT Services'
}

Exchange_map = {
    'NATIONAL STOCK EXCHANGE':'NSE'
}
cleaned_df['sector'] = cleaned_df['sector'].replace(Sector_map)
cleaned_df['exchange'] = cleaned_df['exchange'].replace(Exchange_map)

In [40]:
cleaned_df['sector'].unique()

<StringArray>
['Private Bank', 'Public Sector Bank', 'IT Services', 'Small Finance Bank']
Length: 4, dtype: str

In [41]:
cleaned_df['exchange'].unique()

<StringArray>
['NSE']
Length: 1, dtype: str

In [42]:
cleaned_df[['sma_20','rsi_14','analyst_rating']].dtypes

sma_20            float64
rsi_14            float64
analyst_rating    float64
dtype: object

In [43]:
cols = ['sma_20','rsi_14','analyst_rating']
print((cleaned_df[cols] == -1).sum())
mask = (cleaned_df[cols] == -1).any(axis=1)
print(cleaned_df[mask].head(10))

before = cleaned_df[cols].isna().sum()
cleaned_df[cols] = cleaned_df[cols].replace(-1,np.nan)

after = cleaned_df[cols].isna().sum()

print(after - before)

sma_20            831
rsi_14              0
analyst_rating    979
dtype: int64
     trade_id  company_id             company_name              sector  \
25     317367        5018   Persistent Systems Ltd         IT Services   
31     309866        5010     Punjab National Bank  Public Sector Bank   
51     300921        5000            HDFC Bank Ltd        Private Bank   
64     302259        5002  Kotak Mahindra Bank Ltd        Private Bank   
67     301702        5001           ICICI Bank Ltd        Private Bank   
78     316890        5017          LTIMindtree Ltd         IT Services   
90     312270        5012              Infosys Ltd         IT Services   
104    307487        5007         Bandhan Bank Ltd        Private Bank   
108    307708        5008      State Bank of India  Public Sector Bank   
114    311672        5012              Infosys Ltd         IT Services   

    exchange  index_name market_cap_segment  close_price  holding_period_days  \
25       NSE    nifty it 

In [44]:
cleaned_df[['sma_20','rsi_14','analyst_rating','open_interest_lots']].describe()

,sma_20,rsi_14,analyst_rating,open_interest_lots
count,18127.000000,19764.000000,17845.000000,18376.000000
mean,2118.203271,49.913338,3.796474,29987.482314
std,2298.927980,20.625228,0.496899,17263.386140
min,59.560000,5.000000,1.750000,500.000000
25%,544.480000,35.527500,3.470000,14914.000000
50%,1397.080000,49.930000,3.790000,29956.500000
75%,2985.285000,64.270000,4.140000,44978.750000
max,12918.060000,95.000000,5.000000,59999.000000


In [45]:
cols = ['sma_20','rsi_14','analyst_rating']
print((cleaned_df[cols] == -1).sum())

sma_20            0
rsi_14            0
analyst_rating    0
dtype: int64


In [46]:
cleaned_df['open_interest_lots'].dtype

dtype('float64')

In [47]:
cleaned_df[['sma_20','rsi_14','analyst_rating','open_interest_lots']].isna().sum()

sma_20                1637
rsi_14                   0
analyst_rating        1919
open_interest_lots    1388
dtype: int64

In [48]:
(cleaned_df['open_interest_lots'] == 0).sum()

np.int64(0)

In [49]:
(cleaned_df['open_interest_lots'] < 0).sum()

np.int64(0)

In [50]:
cleaned_df['open_interest_lots'] = cleaned_df['open_interest_lots'].fillna(0)

In [51]:
cleaned_df[['sma_20','rsi_14','analyst_rating','open_interest_lots']].isna().sum()

sma_20                1637
rsi_14                   0
analyst_rating        1919
open_interest_lots       0
dtype: int64

## Missing values in Technical Indicators('sma_20','rsi_14','analyst_rating','open_interest_lots')
* **Action:** Retained missing values as 'NaN' (did not fill or drop).
* **Why this approach?**
* **Prevents false signals:** sma and rsi are calculated technical indicators.Filling missing values with 0 or averages would create misleading buy / sell momentum signals.
* **Cleaner aggregation:** Leaving them as 'NaN' allows SQL and Power BI to automatically ignore missing values without distorting overall calculations.
* converted -1 placeholders to missing in sma_20/rsi_14/analyst_rating
* **Open_interest_lots:** filled missing values with `0`(equity/cash segment stocks do not trade derivatives).


In [52]:
cleaned_df['circuit_hit'].unique()

<StringArray>
['f', 'F', nan, 'FALSE', 'False', 'True', 't', 'TRUE', 'T']
Length: 9, dtype: str

In [53]:
circuit_map = {
    'f': False, 'F':False, 'FALSE':False,'False':False,
    'True':True, 't':True,'TRUE':True, 'T': True
}
cleaned_df['circuit_hit'] = cleaned_df['circuit_hit'].replace(circuit_map)
cleaned_df['circuit_hit'] = cleaned_df['circuit_hit'].astype('boolean')

In [54]:
cleaned_df['circuit_hit'].unique()

<BooleanArray>
[False, <NA>, True]
Length: 3, dtype: boolean

In [55]:
cleaned_df['circuit_hit'].dtype

BooleanDtype

In [56]:
cleaned_df['circuit_hit'].isnull().sum()

np.int64(755)

### Standardization: `circuit_hit`
* **Action:** Converted mixed text to native boolean,retaining values as `<NA>`.
* **Why this approach?** Standardizes the flag for SQL and Power BI without making false assumptions-filling missing entries with `False` would incorrectly claim no circuit was hit when the status was actually unrecorded.

In [57]:
rsi_condition = (cleaned_df['rsi_14']> 80) | (cleaned_df['rsi_14'] <20 )
atr_condition = (cleaned_df['atr_14'] / cleaned_df['close_price']) > 0.05
cleaned_df['is_outlier'] = np.where(rsi_condition | atr_condition, 1, 0)

In [58]:
cleaned_df['is_outlier'].value_counts()

is_outlier
0    16504
1     3260
Name: count, dtype: int64

### Outlier Analysis Summary
* **Normal Trading Days(`0`)**: 16,496 rows
* **Extreme / Volatile Days (`1`)**: 3,268 rows
* **Actions**: Flagged extreme days using `is_outlier` without deleting any data to keep complete market history.

In [59]:
cleaned_df['momentum_flag'] = np.where(
    cleaned_df['close_price'] > cleaned_df['sma_20'], 'Bullish','Bearish'
)

In [60]:
rsi_conditions = [
    cleaned_df['rsi_14'] <30,
    cleaned_df['rsi_14'] >70
]
rsi_labels =['Oversold','Overbought']

cleaned_df['Rsi_zone'] = np.select(
    rsi_conditions,
    rsi_labels,
    default='Neutral'
)

In [61]:
delivery_conditions=[
    cleaned_df['delivery_pct']<30,
    cleaned_df['delivery_pct']>60
]

delivery_labels =['Low delivery','High delivery']

cleaned_df['delivery_bucket'] = np.select(
    delivery_conditions,
    delivery_labels,
    default='Medium delivery'
)

In [62]:
cleaned_df[['week52_high','week52_low']].isnull().sum()

week52_high    0
week52_low     0
dtype: int64

In [63]:
(cleaned_df['week52_high']<cleaned_df['week52_low']).sum()

np.int64(0)

In [64]:
cleaned_df['pct_from_52w_high'] = (
    (cleaned_df['close_price'] - cleaned_df['week52_high']) / cleaned_df['week52_high'] 
) * 100

In [65]:
cleaned_df['pct_from_52w_low'] = (
    (cleaned_df['close_price'] -cleaned_df['week52_low']) / cleaned_df['week52_low']
) *100

### Business Feature Engineering
Added simple business features for analysis and reporting:
* **momentum flag**: Shows whether price is above or below the 20-day SMA.
* **rsi_zone**: Groups `RSI` into `oversold` , `Neutral` and `overbought`.
* **delivery_bucket**: Groups delivery % into `Low`,`Medium` and `High`.
* **pct_from_52w_high/low**: Shows how far the current price is from the 52-week high and low.

In [66]:
cleaned_df[['last_trade_date','listed_since']].head(20)

,last_trade_date,listed_since
0,2025-08-26,1998-02-19
1,2026-02-23,1995-03-01
2,2025-08-06,1994-01-12
3,2026-05-04,2000-01-06
4,2024-02-20,2006-08-28
5,2026-05-11,1994-01-12
8,2025-05-09,2016-09-23
9,2023-06-05,1993-01-01
10,2026-05-18,1946-09-01
11,2026-08-24,2017-07-10


In [67]:
cleaned_df['last_trade_date'] = pd.to_datetime(cleaned_df['last_trade_date'],errors='coerce')

In [68]:
cleaned_df['listed_since'] = pd.to_datetime(cleaned_df['listed_since'],errors='coerce')

In [69]:
cleaned_df[['last_trade_date','listed_since']].dtypes

last_trade_date    datetime64[us]
listed_since       datetime64[us]
dtype: object

In [70]:
cleaned_df.head()

,trade_id,company_id,company_name,sector,exchange,index_name,market_cap_segment,close_price,holding_period_days,number_of_trades,trades_per_day,trading_days_active_365,week52_high,week52_low,listed_since,last_trade_date,analyst_rating,sma_20,rsi_14,atr_14,open_interest_lots,circuit_hit,settlement_type,delivery_pct,trading_status,data_source,settlement_mode,next_day_direction,is_outlier,momentum_flag,Rsi_zone,delivery_bucket,pct_from_52w_high,pct_from_52w_low
0,304456,5004,IndusInd Bank Ltd,Private Bank,NSE,NIFTY BANK,Mid Cap,913.34,14,40720,7.88,216,1742.85,811.62,1998-02-19,2025-08-26,3.05,866.51,53.52,7.41,15699.0,False,T+1,47.8,Suspended,Moneycontrol,F&O,Up,0,Bullish,Neutral,Medium delivery,-47.595031,12.532959
1,308360,5008,State Bank of India,Public Sector Bank,NSE,nifty bank,Large Cap,638.64,20,28617,NaN,216,800.25,576.29,1995-03-01,2026-02-23,5.00,669.46,44.58,9.20,33293.0,False,Delivery,NaN,Unknown,Nse Bhavcopy,Cash,Down,0,Bearish,Neutral,Medium delivery,-20.194939,10.819206
2,305387,5005,Federal Bank Ltd,Private Bank,NSE,NIFTY BANK,Mid Cap,109.00,22,26687,4.40,213,178.21,102.53,1994-01-12,2025-08-06,4.05,107.10,83.51,0.59,40572.0,False,Delivery,33.8,Unknown,Bloomberg,Cash,Down,1,Bullish,Overbought,Medium delivery,-38.836204,6.310348
3,314078,5014,HCL Technologies Ltd,IT Services,NSE,Nifty It,Large Cap,1470.60,25,24470,11.90,239,1793.92,1227.62,2000-01-06,2026-05-04,4.25,1517.24,69.48,11.48,14095.0,<NA>,T+1,50.7,Unknown,Unknown,F&O,Up,0,Bearish,Neutral,Medium delivery,-18.023100,19.792770
4,315409,5016,Tech Mahindra Ltd,IT Services,NSE,NIFTY IT,Mid Cap,2057.69,19,35973,7.48,248,2720.63,1464.66,2006-08-28,2024-02-20,3.91,2032.40,50.90,7.98,5539.0,False,Btst,43.2,Active,Bloomberg,Cash,Up,0,Bullish,Neutral,Medium delivery,-24.367150,40.489260


In [76]:
cleaned_df = cleaned_df.dropna(subset=['company_name'])

In [77]:
cleaned_df = cleaned_df.reset_index(drop =True)

In [78]:
print(cleaned_df.index)

RangeIndex(start=0, stop=18557, step=1)


In [79]:
print('current total clean rows:',len(cleaned_df))
print('total rows removed overall:',20914 - len(cleaned_df))

current total clean rows: 18557
total rows removed overall: 2357


In [80]:
cleaned_df.to_csv('../data/cleaned_data/clean_stock_market.csv',index=False)